# Demo 2: Controlled Pendulum
## Demo 2.1: OpenModelica with Python

### Description

The following demo shows a simple 1-DOF pendulum model that is driven by a `Drive` to follow a reference trajectory.
The reference trajectory is a sine wave with parameterizable mean, amplitude, and frequency.

The refernence and state pendulum angle are measured by an `AngleEncoder` that acts as a sensor. The encoder is modeled as an potentiometer, followed by an ideal ADC. Noise on the potentiometer and quantization is neglected in this demo.

The quantized output voltage for the reference and state angle is fed into a `Controller` block that implements a PID controller.
The controller outputs a control signal within the range of -1 to 1, which is then fed into the `Drive` block to obtain the required torque to drive the pendulum. The `Drive` also gets as input the angular velocity of the pendulum, which is required to compute the resulting torque.

The `Pendulum` gets as input the torque from the `Drive`. The pendulum equations are integrated to obtain the angular position and velocity of the pendulum. 

### Procedure

**1. Get the OpenModelica package and modelica files**

In [63]:
from path import Path
from OMPython import ModelicaSystem

# Define paths and package information
root = Path(r'/home/flo/repos/SystemSimulation/demos/ControlledPendulum/')
pkg_str = str(root / "ControlledPendulum" / "package.mo")
pkg_name = "ControlledPendulum"
model_name = pkg_name + "." + "Demo_Driven"

# Create and build the model
demo_model = ModelicaSystem(pkg_str, model_name)
demo_model.buildModel()

# Change model parameters
#demo_model.setParameters("reference.frequency=0.33")

# Simulation parameters
current_options = demo_model.getSimulationOptions()
print(f"Simulation options:", current_options)

# Run simulation
demo_model.simulate()

# Extract results
results = demo_model.getSolutions()
t_vals = demo_model.getSolutions('time').flatten()
q_ref_vals = demo_model.getSolutions('reference.q_ref').flatten()
q_state_vals = demo_model.getSolutions('pendulum.q_state').flatten()


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.


Simulation options: {'startTime': '0', 'stopTime': '10', 'stepSize': '0.001', 'tolerance': '1e-06', 'solver': 'dassl', 'outputFormat': 'mat'}
LOG_SUCCESS       | info    | The initialization finished successfully without homotopy method.
LOG_SUCCESS       | info    | The simulation finished successfully.


**2. Generate the Modelica System for Python using OMPython**

In [64]:
from OMPython import ModelicaSystem
import numpy as np

def create_modelica_system(model_name):
    model = ModelicaSystem(pkg_str, model_name, verbose=True)
    return model

model_name = pkg_name + "." + "Demo_Driven"
demo_model = create_modelica_system(model_name)

demo_model.buildModel()
demo_model.setParameters(f"pendulum.q0={-np.pi/4}")


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.




**3. Check the parameters and variables of the system**

In [65]:
# Get involved components and their variables
states = demo_model.getContinuous()

component_dict = {}

for state in states:
    # Check if the state name contains a dot
    if '.' in state:
        component_name, var_name = state.split('.', 1)
        # Remove ( and ) if present
        var_name = var_name.replace('(', '').replace(')', '')
        if component_name not in component_dict:
            component_dict[component_name] = []
        component_dict[component_name].append(var_name)

# Print the components and their variables
for component, variables in component_dict.items():
    print(f"Component: {component}")
    for var in variables:
        print(f"  Variable: {var}")
    print()

Component: drive
  Variable: I
  Variable: U
  Variable: n
  Variable: torque
  Variable: omega
  Variable: u_control

Component: pendulum
  Variable: omega_state
  Variable: q_state
  Variable: torque

Component: pid
  Variable: D.x
  Variable: I.y
  Variable: D.y
  Variable: I.local_set
  Variable: P.y
  Variable: addErr.y
  Variable: lim.simplifiedExpr
  Variable: sumPID.y
  Variable: u
  Variable: D.u
  Variable: I.u
  Variable: P.u
  Variable: addErr.u1
  Variable: addErr.u2
  Variable: gainK.u
  Variable: gainK.y
  Variable: lim.u
  Variable: lim.y
  Variable: ref
  Variable: sumPID.u1
  Variable: sumPID.u2
  Variable: sumPID.u3
  Variable: y

Component: der(drive
  Variable: I

Component: der(pendulum
  Variable: omega_state
  Variable: q_state

Component: der(pid
  Variable: D.x
  Variable: I.y

Component: reference
  Variable: q_ref

Component: sensor_ref
  Variable: U_a
  Variable: U_q
  Variable: alpha
  Variable: q

Component: sensor_state
  Variable: U_a
  Variable: U_q
  

**4. Set the simulation parameters**

In [66]:
parameters = demo_model.getParameters()
default_frequency = parameters.get('reference.frequency')
default_frequency

'0.25'

In [67]:
# Set simulation parameters
simulation_options = [
    "startTime=0.0",
    "stopTime=10.0",        
    "stepSize=0.01",
    "tolerance=1e-6"
]

demo_model.setSimulationOptions(simulation_options)
current_options = demo_model.getSimulationOptions()
print(f"Simulation options:")
for option in current_options:
    print(f"  {option}: {current_options[option]}")

Simulation options:
  startTime: 0.0
  stopTime: 10.0
  stepSize: 0.01
  tolerance: 1e-6
  solver: dassl
  outputFormat: mat


**5. Run the simulation**

In [68]:
# Run simulation
print("Running simulation...")
demo_model.simulate()
print("Simulation completed successfully")

Running simulation...
LOG_SUCCESS       | info    | The initialization finished successfully without homotopy method.
LOG_SUCCESS       | info    | The simulation finished successfully.
Simulation completed successfully


**6. Extract and plot the results**

In [79]:
# Extract results
results = demo_model.getSolutions()

t_vals = demo_model.getSolutions('time')
q_ref_vals = demo_model.getSolutions('reference.q_ref')
q_state_vals = demo_model.getSolutions('pendulum.q_state')
omega_state_vals = demo_model.getSolutions('pendulum.omega_state')

t_vals = t_vals.flatten()
q_ref_vals = q_ref_vals.flatten()
q_state_vals = q_state_vals.flatten()
omega_state_vals = omega_state_vals.flatten()

# Get only every 0.01 s the values of q_state and omega_state
q_state_vals_mot = []
omega_state_vals_mot = []
t = 0.0
for i in range(1000):
    t = 0.01 * i
    index = np.argmin(np.abs(t_vals - t))
    q_state_vals_mot.append(q_state_vals[index])
    omega_state_vals_mot.append(omega_state_vals[index])

In [82]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(
    title={'text': 'Pendulum Angle - Modelica Demo using OMPython', 'font': {'size': 24}},
    xaxis_title={'text': 'Time (s)', 'font': {'size': 20}},
    yaxis_title={'text': 'Angle (rad)', 'font': {'size': 20}},
    legend_title={'text': 'Legend', 'font': {'size': 18}},
    font={'size': 16},
    template='plotly_dark'
)

fig.show()

In [83]:
import os
import sys
from path import Path
repo_root = Path.getcwd().parent.parent
sys.path.insert(0, str(repo_root))

In [85]:
from SysSimX.utilities.results_opensim import create_opensim_mot_file
import numpy as np

t_vals = t_vals[::2]

data = {'q': q_state_vals_mot,
        '/jointset/head_joint/q/speed': omega_state_vals_mot}
time = t_vals
n_time_steps = 1000
filename = 'OpenSim/Results/demo_2_1.mot'

create_opensim_mot_file(data=data, time=time, filename=filename)